# 04. Streaming Runtime

`arcrun.run_stream()` is the live-output sibling of `run()` / `run_async()`. It runs the same agent loop, but instead of returning a single `LoopResult` at the end, it yields a sequence of typed `StreamEvent` objects as the run progresses — token chunks, tool starts, tool ends, and finally a single `TurnEndEvent` that closes the stream.

This is what powers chat UIs, REPLs, and any surface where you want to *show* the agent thinking instead of staring at a spinner.

**You will learn:**

- Why streaming matters — UX latency, partial output, debugging visibility
- How `run_stream()` wraps `run()` and yields events through an `async for` iterator
- The `StreamEvent` hierarchy: `TokenEvent`, `ToolStartEvent`, `ToolEndEvent`, `TurnEndEvent`
- Token accumulation: rebuilding full text from chunks
- Tool lifecycle observation through the stream
- How `task_complete` interacts with streamed runs
- Backpressure semantics — what happens when the consumer is slow
- A live REPL pattern you can drop into a chat UI

## 1. Setup

In [ ]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

Optional live key — only used by the gated `(live)` section near the end. Every other cell runs hermetically with a mock model.

In [ ]:
# (live) optional — set this to run real-API sections; mock cells run regardless
HAS_LIVE_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
print(f"Live API key: {'present' if HAS_LIVE_KEY else 'missing — live cells will skip'}")

Pull the public streaming surface into the namespace. These six imports are the entire API for this notebook.

In [ ]:
from arcrun import (
    StaticProvider,
    StreamEvent,
    TokenEvent,
    ToolStartEvent,
    ToolEndEvent,
    TurnEndEvent,
    run_stream,
)
from arcrun.types import Tool

print("Streaming surface:")
for sym in (StreamEvent, TokenEvent, ToolStartEvent, ToolEndEvent, TurnEndEvent, run_stream):
    print(f"  {sym.__name__:<16} {sym!r}")

We also need a `MockModel`. The arcrun `run()` API expects an object with an async `invoke(messages, tools=...)` method that returns an `arcllm.LLMResponse`. We inline a tiny version here — exactly the same approach used in `01-core-react.ipynb` to keep the notebook free of test-conftest collisions.

In [ ]:
from arcllm.types import LLMResponse, ToolCall, Usage


class MockModel:
    """Replays a scripted list of LLMResponse objects, one per .invoke() call."""

    def __init__(self, responses: list[LLMResponse]) -> None:
        self._responses = list(responses)
        self._idx = 0

    async def invoke(
        self,
        messages: list,
        tools: list | None = None,
        **kwargs,
    ) -> LLMResponse:
        if self._idx >= len(self._responses):
            # Demos shouldn't hang if a script is short — keep ending the turn.
            return _end_turn("(mock exhausted)")
        resp = self._responses[self._idx]
        self._idx += 1
        return resp


def _usage(input_tokens: int = 4, output_tokens: int = 8) -> Usage:
    return Usage(
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        total_tokens=input_tokens + output_tokens,
    )


def _end_turn(text: str) -> LLMResponse:
    return LLMResponse(
        content=text,
        model="mock-1",
        stop_reason="end_turn",
        usage=_usage(),
        cost_usd=0.0,
    )


def _tool_use(tool_name: str, args: dict, call_id: str = "call_1") -> LLMResponse:
    return LLMResponse(
        content=None,
        tool_calls=[ToolCall(id=call_id, name=tool_name, arguments=args)],
        model="mock-1",
        stop_reason="tool_use",
        usage=_usage(),
        cost_usd=0.0,
    )


# arcrun.run() requires a non-empty tool list, so we register a no-op
# 'noop' tool for cells that demonstrate model-only behavior.
async def _noop_exec(args, ctx):
    return ""


noop_tool = Tool(
    name="noop",
    description="A no-op tool — never called by the model.",
    input_schema={"type": "object", "properties": {}, "required": []},
    execute=_noop_exec,
)

print("MockModel ready.")

## 2. Why streaming matters

Without streaming, an agent run is a black box: you call `run()`, wait, and eventually receive a `LoopResult`. For short tasks that's fine. For anything user-facing it's terrible:

- **Latency perception.** A 6-second response feels instantaneous if the first words appear in   ~200 ms. The same 6 seconds with no output feels broken.
- **Partial output.** Even if the run fails halfway through, the user has already received   useful tokens.
- **Debugging visibility.** Tool calls become observable in real time — you can see *which* tool   the agent reached for and *when*, instead of reverse-engineering it from a final transcript.
- **Cancellation surface.** Once the consumer can see what's happening, it's natural to give   them a stop button.

`run_stream()` solves this without requiring the underlying LLM to support server-sent token streaming. Token text is reconstructed from the final `LoopResult.content` and chunked out progressively. That keeps the API model-agnostic: any provider arcllm supports works with `run_stream()` out of the box.

Three concrete contracts are worth memorizing before we go further:

1. **`run_stream()` returns an async iterator.** You consume it with `async for event in stream:`.
2. **`TurnEndEvent` is always the last event.** Every other event type may or may not appear.
3. **Token text is split on word boundaries** (the implementation does `content.split(' ')`).    Re-joining the `.text` attributes of every `TokenEvent` reconstructs the full final response.

## 3. `run_stream` basics

Smallest possible streaming run: a model that returns a one-line answer and ends the turn. Watch the events fly past.

In [ ]:
async def hello_run() -> list[StreamEvent]:
    model = MockModel([_end_turn("the quick brown fox jumps over the lazy dog")])
    stream = await run_stream(
        model=model,
        capabilities=StaticProvider([noop_tool]),
        system_prompt="You are a helpful assistant.",
        task="Tell me a pangram.",
    )
    events: list[StreamEvent] = []
    async for event in stream:
        events.append(event)
    return events


events = await hello_run()
for ev in events:
    print(type(ev).__name__, ev)

A few things to note in the trace above:

- The `TokenEvent` count tracks word boundaries. Nine words → nine `TokenEvent`s.
- The first token has no leading space; subsequent tokens preserve the inter-word space.
- Exactly one `TurnEndEvent` at the end, carrying the **full** `final_text`, the turn count,   the number of tool calls, and the accumulated cost.
- No tools were registered, so no `ToolStartEvent` / `ToolEndEvent` appear.

The signature of `run_stream()` mirrors `run()` closely. Required keyword args: `model`, `capabilities`, `system_prompt`, `task`. Optional: `max_turns`, `sandbox`, `allowed_strategies`, `audit_sink`, `ui_reporter`. Notice the missing `on_event` — the function uses that hook internally to bridge `EventBus` events into the stream.

Like `run()`/`run_async()` (see `arcrun/01-core-react.ipynb`), `run_stream()` takes a `capabilities: CapabilityProvider` (ADR-023), not a raw `tools=[...]` list. `StaticProvider(tools)` adapts a fixed tool list to that contract — used throughout this notebook.

In [ ]:
import inspect

sig = inspect.signature(run_stream)
print("run_stream signature:")
for name, param in sig.parameters.items():
    print(f"  {name:<18} {param.kind.name:<20} default={param.default!r}")

## 4. The `StreamEvent` hierarchy

All four event subclasses share a common base. They are plain `@dataclass` types — no Pydantic, no validation overhead, just typed shapes you can pattern-match on.

| Class | Fields | Emitted when |
|---|---|---|
| `StreamEvent` | (none) | Never directly — abstract base |
| `TokenEvent` | `text: str` | A chunk of the model's response text |
| `ToolStartEvent` | `name: str`, `args: dict[str, Any]` | The model has decided to call a tool |
| `ToolEndEvent` | `result: str` | A tool returned (or errored — the result is stringified) |
| `TurnEndEvent` | `final_text: str`, `turns: int`, `tool_calls_made: int`, `cost_usd: float` | Always last; closes the stream |

In [ ]:
from dataclasses import fields

for cls in (StreamEvent, TokenEvent, ToolStartEvent, ToolEndEvent, TurnEndEvent):
    parents = " < ".join(b.__name__ for b in cls.__mro__[:3])
    field_repr = ", ".join(f"{f.name}: {f.type}" for f in fields(cls))
    print(f"{cls.__name__:<18} {parents}\n  fields: {field_repr or '(none)'}\n")

Because everything inherits from `StreamEvent`, you can write generic consumers that handle the base type and `isinstance`-narrow only when they need to. This is the pattern most production consumers use.

In [ ]:
def classify(event: StreamEvent) -> str:
    if isinstance(event, TokenEvent):
        return f"token={event.text!r}"
    if isinstance(event, ToolStartEvent):
        return f"tool_start name={event.name} args={event.args}"
    if isinstance(event, ToolEndEvent):
        return f"tool_end result={event.result!r}"
    if isinstance(event, TurnEndEvent):
        return (
            f"turn_end turns={event.turns} tool_calls={event.tool_calls_made} "
            f"cost_usd={event.cost_usd}"
        )
    return f"unknown {type(event).__name__}"


for ev in events:
    print(classify(ev))

## 5. `TokenEvent` — token-by-token streaming

Every `TokenEvent` carries exactly one `.text` chunk. Concatenating every chunk in order gives you the final response text. This invariant is locked down by the test suite — it's safe to depend on.

In [ ]:
async def accumulate_run(answer: str) -> tuple[str, str, list[str]]:
    model = MockModel([_end_turn(answer)])
    stream = await run_stream(
        model=model,
        capabilities=StaticProvider([noop_tool]),
        system_prompt="sys",
        task="task",
    )
    chunks: list[str] = []
    final_text: str = ""
    async for event in stream:
        if isinstance(event, TokenEvent):
            chunks.append(event.text)
        elif isinstance(event, TurnEndEvent):
            final_text = event.final_text
    return "".join(chunks), final_text, chunks


answer = "Streaming reduces perceived latency by surfacing partial output."
rebuilt, final_text, chunks = await accumulate_run(answer)

print(f"chunks  : {chunks}")
print(f"rebuilt : {rebuilt!r}")
print(f"final   : {final_text!r}")
print(f"match   : {rebuilt == final_text == answer}")

**Invariant under test:** `''.join(e.text for e in token_events) == TurnEndEvent.final_text`. If you write a renderer that appends each chunk to a buffer as it arrives, the buffer at `TurnEndEvent` time matches the final answer exactly. No reconciliation step needed.

Two practical accumulator patterns. The first is a stateful renderer that prints incremental output without buffering — what a chat UI does. The second is a buffer-and-emit-paragraphs filter for surfaces that want chunky updates instead of letter-perfect ones.

In [ ]:
async def stream_to_print(model_response: str) -> None:
    """Pattern A: print each chunk as it arrives — a chat-UI streaming renderer."""
    model = MockModel([_end_turn(model_response)])
    stream = await run_stream(
        model=model,
        capabilities=StaticProvider([noop_tool]),
        system_prompt="sys",
        task="task",
    )
    async for event in stream:
        if isinstance(event, TokenEvent):
            print(event.text, end="", flush=True)
        elif isinstance(event, TurnEndEvent):
            print()  # newline after final token


await stream_to_print("Token streaming makes the agent feel alive.")

In [ ]:
async def stream_paragraphs(model_response: str) -> list[str]:
    """Pattern B: buffer until a sentence boundary, then emit."""
    model = MockModel([_end_turn(model_response)])
    stream = await run_stream(
        model=model,
        capabilities=StaticProvider([noop_tool]),
        system_prompt="sys",
        task="task",
    )
    paragraphs: list[str] = []
    buf = ""
    async for event in stream:
        if isinstance(event, TokenEvent):
            buf += event.text
            if buf.endswith((".", "!", "?")):
                paragraphs.append(buf.strip())
                buf = ""
        elif isinstance(event, TurnEndEvent):
            if buf.strip():
                paragraphs.append(buf.strip())
    return paragraphs


para_text = "First sentence is short. Second sentence is also short. Third sentence wraps it up."
paragraphs = await stream_paragraphs(para_text)
for i, p in enumerate(paragraphs):
    print(f"para {i}: {p!r}")

## 6. `ToolStartEvent` / `ToolEndEvent` — the tool lifecycle

When the model emits a tool call, `run_stream()` surfaces two events around the dispatch:

1. `ToolStartEvent(name=..., args=...)` — *before* the tool runs. This is your hook for   showing a spinner, logging an audit note, or warning a human reviewer.
2. `ToolEndEvent(result=...)` — *after* the tool returns. The result is stringified — even if   the underlying tool returns a structured object, you see its `str()` form.

Both events ride alongside any token output. Note that token events are emitted only at the *end* of the loop (from the final `LoopResult.content`), so during a multi-turn run the order is: tool events live → tokens at the end → `TurnEndEvent`.

In [ ]:
from typing import Any


async def echo_exec(args: dict[str, Any], ctx: Any) -> str:
    return f"echo: {args.get('msg', '')}"


echo_tool = Tool(
    name="echo",
    description="Echo a message back to the caller.",
    input_schema={
        "type": "object",
        "properties": {"msg": {"type": "string"}},
        "required": ["msg"],
    },
    execute=echo_exec,
)


async def tool_run() -> list[StreamEvent]:
    model = MockModel(
        [
            _tool_use("echo", {"msg": "ping"}),
            _end_turn("Done — the echo returned the expected reply."),
        ]
    )
    stream = await run_stream(
        model=model,
        capabilities=StaticProvider([echo_tool]),
        system_prompt="Use the echo tool when asked.",
        task='Echo "ping".',
    )
    out: list[StreamEvent] = []
    async for event in stream:
        out.append(event)
    return out


tool_events = await tool_run()
for ev in tool_events:
    print(f"{type(ev).__name__:<16} {ev}")

Notice the order: `ToolStartEvent` and `ToolEndEvent` come *before* any `TokenEvent`. That's because token reconstruction happens after the loop completes — by the time we know what the final text is, the tool calls have already happened. If you need a richer interleaving (token chunks that genuinely interleave with tool calls in real time), that requires LLM-native streaming and a future revision of `streams.py`.

Counting tools is a one-liner. The number must always equal `TurnEndEvent.tool_calls_made`.

In [ ]:
tool_starts = [e for e in tool_events if isinstance(e, ToolStartEvent)]
tool_ends = [e for e in tool_events if isinstance(e, ToolEndEvent)]
turn_end = next(e for e in tool_events if isinstance(e, TurnEndEvent))

print(f"tool_starts in stream: {len(tool_starts)}")
print(f"tool_ends   in stream: {len(tool_ends)}")
print(f"turn_end.tool_calls_made: {turn_end.tool_calls_made}")
assert len(tool_starts) == len(tool_ends) == turn_end.tool_calls_made
print("invariant holds.")

## 7. `TurnEndEvent` — the final boundary

`TurnEndEvent` is special. It is the **only** event guaranteed to appear in every stream — even if the model returns empty content, even if no tools fire, the stream still closes with exactly one `TurnEndEvent`. Treat it as your synchronization point.

Its four fields:

- `final_text` — the complete model response. Equals `''.join(t.text for t in tokens)`.
- `turns` — turn count from `LoopResult`.
- `tool_calls_made` — equals the number of `ToolStartEvent`s in this stream.
- `cost_usd` — accumulated cost from the underlying LLM calls (0.0 for mocks).

This is the right place to hand the run off to whatever lives after the stream — an audit summary, a UI badge update, a downstream pipeline trigger.

In [ ]:
async def empty_run() -> list[StreamEvent]:
    """Even with empty content, TurnEndEvent must appear."""
    model = MockModel([_end_turn("")])
    stream = await run_stream(
        model=model,
        capabilities=StaticProvider([noop_tool]),
        system_prompt="sys",
        task="task",
    )
    out: list[StreamEvent] = []
    async for event in stream:
        out.append(event)
    return out


empty_events = await empty_run()
print("event types:", [type(e).__name__ for e in empty_events])
assert isinstance(empty_events[-1], TurnEndEvent)
assert empty_events[-1].final_text == ""
print(f"final_text: {empty_events[-1].final_text!r}")
print("invariant holds: TurnEndEvent always closes the stream.")

Practical pattern: render incrementally, then commit on `TurnEndEvent`. This is exactly what a chat-UI message component does — show partial state while it's streaming, then mark the message `done` once the close event arrives.

In [ ]:
class StreamingMessage:
    """Mimics a chat-UI message component lifecycle."""

    def __init__(self) -> None:
        self.body: str = ""
        self.done: bool = False
        self.cost_usd: float = 0.0
        self.tool_calls: int = 0

    def push(self, event: StreamEvent) -> None:
        if isinstance(event, TokenEvent):
            self.body += event.text
        elif isinstance(event, ToolStartEvent):
            self.tool_calls += 1
        elif isinstance(event, TurnEndEvent):
            self.body = event.final_text
            self.cost_usd = event.cost_usd
            self.done = True


async def render_one() -> StreamingMessage:
    model = MockModel([_end_turn("Streaming chat UIs commit on TurnEndEvent.")])
    stream = await run_stream(
        model=model,
        capabilities=StaticProvider([noop_tool]),
        system_prompt="sys",
        task="task",
    )
    msg = StreamingMessage()
    async for event in stream:
        msg.push(event)
    return msg


msg = await render_one()
print(f"body : {msg.body!r}")
print(f"done : {msg.done}")
print(f"cost : ${msg.cost_usd:.5f}")

## 8. Composition with `task_complete`

`task_complete` (from `arcrun.builtins`) is a *terminator tool* — when the model invokes it, the loop stops, and the tool's call arguments become the structured completion payload. Streaming does not change this behavior; you simply observe the same lifecycle through the stream:

- `ToolStartEvent(name='task_complete', args={...})` fires.
- The loop exits *immediately* after that tool runs (no second model turn).
- `TurnEndEvent` appears with whatever final text the loop accumulated and `tool_calls_made=1`.

See `arcrun/06-task-completion-budgets.ipynb` for the full deep-dive on `TaskCompleteArgs` and budget enforcement. Here we only verify that early termination flows cleanly through the stream API.

In [ ]:
from arcrun.builtins import make_task_complete_tool

tc_tool = make_task_complete_tool()
print(f"tool name: {tc_tool.name}")
print(f"signals_completion: {tc_tool.signals_completion}")

In [ ]:
async def early_complete() -> list[StreamEvent]:
    model = MockModel(
        [
            _tool_use(
                "task_complete",
                {
                    "status": "success",
                    "summary": "Stream-friendly early termination.",
                },
            ),
        ]
    )
    stream = await run_stream(
        model=model,
        capabilities=StaticProvider([tc_tool]),
        system_prompt="Use task_complete when finished.",
        task="Finish quickly.",
    )
    out: list[StreamEvent] = []
    async for event in stream:
        out.append(event)
    return out


tc_events = await early_complete()
for ev in tc_events:
    print(f"{type(ev).__name__:<16} {ev}")

Even on early completion, the contract holds: exactly one `TurnEndEvent`, last in the stream, with `tool_calls_made` matching the number of `ToolStartEvent`s observed.

## 9. Backpressure — what if the consumer is slow?

`run_stream()` is built on `asyncio.Queue`. Internally:

1. The agent loop runs in a background task (`asyncio.ensure_future(_run_loop())`).
2. The loop's `on_event` hook calls `queue.put_nowait(...)` for each tool event.
3. The async generator pulls from the queue with `await queue.get()`.
4. After the loop finishes, the generator emits token events synchronously, then   `TurnEndEvent`, then closes.

The queue is **unbounded**. There is no built-in backpressure — `put_nowait` never blocks, even if the consumer hasn't called `__anext__` for a long time. In practice this is fine for agent runs (event volume is small) but worth knowing if you're building a high-volume fan-out.

Practical implication: if your consumer is slow, **events buffer in memory** until the consumer drains them. The stream will not stall the loop, but RAM grows with the lag.

Demonstration: a deliberately slow consumer. We sleep on every event but the run still completes — events simply queue up while we lag.

In [ ]:
import asyncio
import time


async def slow_consumer(per_event_delay_s: float) -> tuple[float, int]:
    answer = "one two three four five six seven eight nine ten"
    model = MockModel([_end_turn(answer)])
    stream = await run_stream(
        model=model,
        capabilities=StaticProvider([noop_tool]),
        system_prompt="sys",
        task="task",
    )
    started = time.perf_counter()
    received = 0
    async for _ev in stream:
        await asyncio.sleep(per_event_delay_s)
        received += 1
    return time.perf_counter() - started, received


elapsed_fast, n_fast = await slow_consumer(0.0)
elapsed_slow, n_slow = await slow_consumer(0.01)

print(f"fast consumer: {n_fast} events in {elapsed_fast * 1000:.1f} ms")
print(f"slow consumer: {n_slow} events in {elapsed_slow * 1000:.1f} ms")
print("same event count — slow consumer just took longer; nothing dropped.")

**Reading the queue semantics from `streams.py`:**

```python
queue: asyncio.Queue[StreamEvent | None] = asyncio.Queue()
...
queue.put_nowait(ToolStartEvent(...))
```

`asyncio.Queue()` with no `maxsize` is unbounded — the third pillar of the contract.

If you need bounded buffering (drop, throttle, or apply real backpressure), wrap the stream yourself: pull from `run_stream`, push to a bounded `asyncio.Queue(maxsize=N)`, and let the intermediate buffer apply backpressure to *your* layer instead. arcrun deliberately keeps the core stream simple — backpressure policy is an application concern.

Sketch of a bounded buffer wrapper. Useful when you fan a stream out to a UI thread that renders frame-by-frame and must not pile up unbounded work.

In [ ]:
import asyncio
from collections.abc import AsyncIterator


async def bounded(
    source: AsyncIterator[StreamEvent],
    *,
    capacity: int = 8,
) -> AsyncIterator[StreamEvent]:
    """Re-emit events through a bounded queue. Slows the producer if needed."""
    out: asyncio.Queue[StreamEvent | None] = asyncio.Queue(maxsize=capacity)

    async def _pump() -> None:
        async for ev in source:
            await out.put(ev)
        await out.put(None)

    pump_task = asyncio.create_task(_pump())
    try:
        while True:
            item = await out.get()
            if item is None:
                break
            yield item
    finally:
        await pump_task


async def use_bounded() -> int:
    answer = "alpha bravo charlie delta echo foxtrot golf hotel"
    model = MockModel([_end_turn(answer)])
    stream = await run_stream(
        model=model,
        capabilities=StaticProvider([noop_tool]),
        system_prompt="sys",
        task="task",
    )
    count = 0
    async for _ev in bounded(stream, capacity=2):
        count += 1
    return count


count = await use_bounded()
_print = f"events seen through bounded(capacity=2): {count}"
print(_print)

## 10. Live REPL pattern

Putting it all together: a streaming chat-style loop. Each user prompt becomes one `run_stream` call; tokens print as they arrive; tool events render as side annotations; the message finalizes on `TurnEndEvent`.

The pseudocode below is exactly the shape a real terminal REPL or web UI would take. The mock produces a tool call followed by a final response, mirroring how a real model might call a tool then explain what it did.

In [ ]:
async def repl_turn(user_prompt: str) -> dict:
    model = MockModel(
        [
            _tool_use("echo", {"msg": user_prompt}),
            _end_turn(f"I echoed your message ({user_prompt!r}) and got the expected reply."),
        ]
    )
    stream = await run_stream(
        model=model,
        capabilities=StaticProvider([echo_tool]),
        system_prompt="Use the echo tool then summarize.",
        task=user_prompt,
    )

    summary = {
        "tools_used": [],
        "tool_results": [],
        "body": "",
        "cost_usd": 0.0,
    }

    print(f"\n[user] {user_prompt}")
    print("[agent] ", end="", flush=True)
    async for event in stream:
        if isinstance(event, ToolStartEvent):
            summary["tools_used"].append(event.name)
            print(f"\n  ↪ calling {event.name}({event.args})", flush=True)
        elif isinstance(event, ToolEndEvent):
            summary["tool_results"].append(event.result)
            print(f"  ↩ result: {event.result}")
            print("[agent] ", end="", flush=True)
        elif isinstance(event, TokenEvent):
            print(event.text, end="", flush=True)
            summary["body"] += event.text
        elif isinstance(event, TurnEndEvent):
            summary["cost_usd"] = event.cost_usd
            print()  # newline after final token
    return summary


summary = await repl_turn("are you alive?")
print()
print(f"summary: {summary}")

That's the entire shape of a streaming agent UI: one async loop, four event types, no explicit waiting. Wrap `repl_turn` in a `while True: prompt = input(...)` and you have a terminal REPL; replace `print` with WebSocket sends and you have a browser chat.

## 11. (live) optional — real provider streaming

If you have an `ANTHROPIC_API_KEY`, the cell below runs `run_stream()` against a real model. It uses arcllm's `load_model()` registry to construct the provider — the same code path you'd use in production.

Without a key, the cell raises `SystemExit(0)` so notebook execution still passes.

In [ ]:
if not HAS_LIVE_KEY:
    print("skip — set ANTHROPIC_API_KEY to run this section")
    raise SystemExit

from arcllm.registry import load_model

live_model = load_model(
    provider="anthropic",
    model="claude-sonnet-4-6",
)


async def live_stream() -> None:
    stream = await run_stream(
        model=live_model,
        capabilities=StaticProvider([noop_tool]),
        system_prompt="You are concise. Answer in one sentence.",
        task="Summarize what an agent loop does.",
    )
    print("(live) streaming response:")
    async for event in stream:
        if isinstance(event, TokenEvent):
            print(event.text, end="", flush=True)
        elif isinstance(event, TurnEndEvent):
            print(f"\n[turns={event.turns} cost=${event.cost_usd:.5f}]")


await live_stream()

## 12. Summary

`run_stream()` is the live-output API for arcrun. It wraps `run()` with an `asyncio.Queue` bridge and yields a typed sequence of `StreamEvent`s — tokens, tool starts, tool ends, and a single closing `TurnEndEvent`.

**Key takeaways:**

- Consumption pattern: `async for event in stream: ...` — that is the entire API.
- `TurnEndEvent` is always the last event, *every* time. Sync to it.
- Token text is reconstructed by splitting `LoopResult.content` on spaces — concatenating   every `TokenEvent.text` rebuilds the full response exactly.
- Tool events ride alongside tokens but in the current implementation appear *before* tokens   (token reconstruction runs after the loop completes).
- `task_complete` works the same as in non-streaming runs — it just terminates earlier and   the stream closes cleanly.
- The internal queue is unbounded; backpressure is an application concern. Wrap with a   bounded buffer if you need throttling.

**Public API surface covered:** `run_stream`, `StreamEvent`, `TokenEvent`, `ToolStartEvent`, `ToolEndEvent`, `TurnEndEvent`.

**Next:** see `arcrun/05-parallel-dispatch.ipynb` for how the loop dispatches read-only tool batches concurrently, and `arcrun/06-task-completion-budgets.ipynb` for the structured completion + budget enforcement that this notebook touched only briefly.